# 02. LangChain RAG 구축
**Day 5**

> Day 4 **03 RAG** 에서 직접 짠 `split_text` · `search` 를  
> LangChain **Document → Splitter → Embedding → VectorStore** 로 바꿉니다.

**데이터:** `../4일차_찐/samples/pdf_samples/학칙.pdf` (이후 03에서 전체 PDF 확장)

---
## 0. 설치

In [1]:
!uv add PyMuPDF pypdf langchain langchain_community

Resolved 235 packages in 3ms
Checked 210 packages in 27ms


---
## 1. Day 4 RAG vs LangChain RAG

| Day 4 (직접 구현) | LangChain |
|------------------|-----------|
| `pdf_to_text` | `PyMuPDFLoader` |
| `split_text` | `RecursiveCharacterTextSplitter` |
| `tokenize` + 교집합 | `OpenAIEmbeddings` + 벡터 유사도 |
| `INDEX` 리스트 | `FAISS` VectorStore |

---
## 2. Document 로드

In [2]:
from langchain_community.document_loaders import PyPDFLoader
import os 
# PDF 파일을 읽어서 텍스트 데이터를 추출합니다.
pdf_path = os.path.join(os.getcwd(),'samples/rag_samples/학칙.pdf')
loader = PyPDFLoader(pdf_path)

C:\Users\Admin\AppData\Local\Temp\ipykernel_10728\1660011071.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [8]:
import pymupdf
from langchain_core.documents import Document
def load_pdf(path: str) -> list[Document]:
    doc = pymupdf.open(path)
    return [
        Document(page_content=page.get_text(), metadata={"source": path, "page": i})
        for i, page in enumerate(doc)
    ]
loader_2 = load_pdf(pdf_path)

In [5]:
loader

In [6]:
data_nyc = loader.load()
print(data_nyc)

[Document(metadata={'producer': 'Hancom PDF 1.3.0.550', 'creator': 'Hwp 2024 13.0.0.3457', 'creationdate': '2026-06-17T16:37:51+09:00', 'author': 'DR', 'moddate': '2026-06-17T16:37:51+09:00', 'pdfversion': '1.4', 'source': 'd:\\autornd\\SK Autonomous R&D\\실습\\5일차\\samples/rag_samples/학칙.pdf', 'total_pages': 35, 'page': 0, 'page_label': '1'}, page_content='Ⅰ. 대학원 학칙  /  7\nⅠ. 대학원 학칙제정: 1974. 05. 18.개정(제122차): 2026. 06. 06.제1장 총칙제1조(목적) 대학원은 기독교 정신을 바탕으로 하여 창의적 이론과 과학적 방법을 탐구하고, 지도적 인격을 도야하여 인류 문화 향상에 기여함을 목적으로 한다.제2장 과정 및 정원제2조(과정) 대학원에는 석사학위를 취득하기 위한 석사학위과정, 박사학위를 취득하기 위한 박사학위과정, 석사학위과정을 거치지 않고 박사학위를 취득하기 위하여 석사학위과정과 박사학위과정이 통합된 과정(석·박사 통합과정, 이하 “통합과정”)을 두며, 학위과정 외에 학위를 수여하지 아니하는 연구과정을 둘 수 있다.제2조의2(협동과정) ① 대학원에 두는 학위과정으로 학과 외에 두 개 이상의 학과가 공동으로 설치·운영하는 협동과정(이하 “학과 간 협동과정”이라 한다)과 연구기관 또는 산업체와의 계약에 의하여 설치·운영하는 학·연·산 협동과정(이하 “학·연·산 협동과정”이라 한다)을 둘 수 있다.② 대학원장은 학과 간 협동과정의 운영실적을 2년마다 평가하여 대학원운영위원회 의결을 거쳐 존속여부를 결정하며, 이에 관한 사항은 따로 정한다.③ <삭제> 제2조의3 <삭제>제2조의4 <삭제>제2조의5(연계과정) 대학원에 두는 학위과정으로 본교 학부와

In [11]:
print(loader_2)

[Document(metadata={'source': 'd:\\autornd\\SK Autonomous R&D\\실습\\5일차\\samples/rag_samples/학칙.pdf', 'page': 0}, page_content='Ⅰ. 대학원 학칙  /  7\nⅠ. 대학원 학칙\n제정: 1974. 05. 18.\n개정(제122차): 2026. 06. 06.\n제1장 총칙\n제1조(목적) 대학원은 기독교 정신을 바탕으로 하여 창의적 이론과 과학적 방법을 탐\n구하고, 지도적 인격을 도야하여 인류 문화 향상에 기여함을 목적으로 한다.\n제2장 과정 및 정원\n제2조(과정) 대학원에는 석사학위를 취득하기 위한 석사학위과정, 박사학위를 취득하기 \n위한 박사학위과정, 석사학위과정을 거치지 않고 박사학위를 취득하기 위하여 석사\n학위과정과 박사학위과정이 통합된 과정(석·박사 통합과정, 이하 “통합과정”)을 두며, \n학위과정 외에 학위를 수여하지 아니하는 연구과정을 둘 수 있다.\n제2조의2(협동과정) ① 대학원에 두는 학위과정으로 학과 외에 두 개 이상의 학과가 \n공동으로 설치·운영하는 협동과정(이하 “학과 간 협동과정”이라 한다)과 연구기관 또\n는 산업체와의 계약에 의하여 설치·운영하는 학·연·산 협동과정(이하 “학·연·산 협동\n과정”이라 한다)을 둘 수 있다.\n② 대학원장은 학과 간 협동과정의 운영실적을 2년마다 평가하여 대학원운영위원회 \n의결을 거쳐 존속여부를 결정하며, 이에 관한 사항은 따로 정한다.\n③ <삭제> \n제2조의3 <삭제>\n제2조의4 <삭제>\n제2조의5(연계과정) 대학원에 두는 학위과정으로 본교 학부와 대학원을 연계하는 학부\n-대학원 연계과정을 둔다. 연계과정 운영에 관한 사항은 따로 정한다. \n제2조의6(계약학과) ① 대학원에 두는 학위과정으로 국가, 지방자치단체 또는 산업체 \n등과의 계약에 의한 학과를 둘 수 있으며, 이의 운영에 관한 사항은 따로 정한다.\n② 대학원장은 대학원운영위원회 의결을 거쳐 계약학과의 존속 여부 등을 

---
## 3. Text Split (청킹)

In [17]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 데이터를 1000자 단위로 나눕니다. overlap은 100자로 설정합니다.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

In [18]:
all_splits = text_splitter.split_documents(data_nyc)

In [20]:
print(type(all_splits[0]))

<class 'langchain_core.documents.base.Document'>


In [21]:
all_splits

[Document(metadata={'producer': 'Hancom PDF 1.3.0.550', 'creator': 'Hwp 2024 13.0.0.3457', 'creationdate': '2026-06-17T16:37:51+09:00', 'author': 'DR', 'moddate': '2026-06-17T16:37:51+09:00', 'pdfversion': '1.4', 'source': 'd:\\autornd\\SK Autonomous R&D\\실습\\5일차\\samples/rag_samples/학칙.pdf', 'total_pages': 35, 'page': 0, 'page_label': '1'}, page_content='Ⅰ. 대학원 학칙  /  7\nⅠ. 대학원 학칙제정: 1974. 05. 18.개정(제122차): 2026. 06. 06.제1장 총칙제1조(목적) 대학원은 기독교 정신을 바탕으로 하여 창의적 이론과 과학적 방법을 탐구하고, 지도적 인격을 도야하여 인류 문화 향상에 기여함을 목적으로 한다.제2장 과정 및 정원제2조(과정) 대학원에는 석사학위를 취득하기 위한 석사학위과정, 박사학위를 취득하기 위한 박사학위과정, 석사학위과정을 거치지 않고 박사학위를 취득하기 위하여 석사학위과정과 박사학위과정이 통합된 과정(석·박사 통합과정, 이하 “통합과정”)을 두며, 학위과정 외에 학위를 수여하지 아니하는 연구과정을 둘 수 있다.제2조의2(협동과정) ① 대학원에 두는 학위과정으로 학과 외에 두 개 이상의 학과가 공동으로 설치·운영하는 협동과정(이하 “학과 간 협동과정”이라 한다)과 연구기관 또는 산업체와의 계약에 의하여 설치·운영하는 학·연·산 협동과정(이하 “학·연·산 협동과정”이라 한다)을 둘 수 있다.② 대학원장은 학과 간 협동과정의 운영실적을 2년마다 평가하여 대학원운영위원회 의결을 거쳐 존속여부를 결정하며, 이에 관한 사항은 따로 정한다.③ <삭제> 제2조의3 <삭제>제2조의4 <삭제>제2조의5(연계과정) 대학원에 두는 학위과정으로 본교 학부와

In [22]:
for i, split in enumerate(all_splits):
    print(f"Split {i+1}:------------------------------------\n")
    print(split)

Split 1:------------------------------------

page_content='Ⅰ. 대학원 학칙  /  7
Ⅰ. 대학원 학칙제정: 1974. 05. 18.개정(제122차): 2026. 06. 06.제1장 총칙제1조(목적) 대학원은 기독교 정신을 바탕으로 하여 창의적 이론과 과학적 방법을 탐구하고, 지도적 인격을 도야하여 인류 문화 향상에 기여함을 목적으로 한다.제2장 과정 및 정원제2조(과정) 대학원에는 석사학위를 취득하기 위한 석사학위과정, 박사학위를 취득하기 위한 박사학위과정, 석사학위과정을 거치지 않고 박사학위를 취득하기 위하여 석사학위과정과 박사학위과정이 통합된 과정(석·박사 통합과정, 이하 “통합과정”)을 두며, 학위과정 외에 학위를 수여하지 아니하는 연구과정을 둘 수 있다.제2조의2(협동과정) ① 대학원에 두는 학위과정으로 학과 외에 두 개 이상의 학과가 공동으로 설치·운영하는 협동과정(이하 “학과 간 협동과정”이라 한다)과 연구기관 또는 산업체와의 계약에 의하여 설치·운영하는 학·연·산 협동과정(이하 “학·연·산 협동과정”이라 한다)을 둘 수 있다.② 대학원장은 학과 간 협동과정의 운영실적을 2년마다 평가하여 대학원운영위원회 의결을 거쳐 존속여부를 결정하며, 이에 관한 사항은 따로 정한다.③ <삭제> 제2조의3 <삭제>제2조의4 <삭제>제2조의5(연계과정) 대학원에 두는 학위과정으로 본교 학부와 대학원을 연계하는 학부-대학원 연계과정을 둔다. 연계과정 운영에 관한 사항은 따로 정한다. 제2조의6(계약학과) ① 대학원에 두는 학위과정으로 국가, 지방자치단체 또는 산업체 등과의 계약에 의한 학과를 둘 수 있으며, 이의 운영에 관한 사항은 따로 정한다.② 대학원장은 대학원운영위원회 의결을 거쳐 계약학과의 존속 여부 등을 결정하며, 이에 관한 사항은 따로 정한다.' metadata={'producer': 'Hancom PDF 1.3.0.550', 'creator': 'Hwp 2024 13.0.0.3457', 'creationd

In [23]:
print(all_splits[50].page_content)
print('----------------------')
print(all_splits[51].page_content)

Ⅰ. 대학원 학칙  /  31
석사학위과정박사학위과정대학학과대학학과
공과대학
항공우주공학과
공과대학
항공우주공학과기후변화에너지융합기술협동과정(모집종료) 기후변화에너지융합기술협동과정기술정책협동과정기술정책협동과정
계약학과
국방융합공학협동과정(폐과존치)
계약학과
국방융합공학협동과정(폐과존치)융합기술경영공학협동과정(폐과존치)반도체데이터사이언스학과이차전지융합공학과이차전지융합공학과우주국방융합학과우주국방융합학과디스플레이융합공학과디스플레이융합공학과시스템반도체공학과시스템반도체공학과디지털융합엔지니어링학과지능형데이터·최적화학과AI인베스트먼트공학과AI인베스트먼트공학과인공위성시스템학과인공위성시스템학과생명시스템대학생명과학부시스템생물학생명시스템대학생명과학부시스템생물학생화학생화학생명공학생명공학융합오믹스·의생명과학협동과정융합오믹스·의생명과학협동과정바이오산업공학협동과정바이오산업공학협동과정인공지능융합대학컴퓨터과학과인공지능융합대학컴퓨터과학과인공지능학과인공지능학과디지털애널리틱스융합협동과정계약학과모빌리티시스템융합학과지능융합학과신과대학신학과신과대학신학과
사회과학대학정치학과사회과학대학정치학과행정학과행정학과사회학과사회학과문화인류학과문화인류학과언론홍보영상학과언론홍보영상학과지역학협동과정지역학협동과정통일학협동과정통일학협동과정사회복지정책협동과정(구)법과대학법학과(구)법과대학법학과음악대학음악학과음악대학음악학과
----------------------
32
석사학위과정박사학위과정대학학과대학학과생활과학대학의류환경학과생활과학대학의류환경학과식품영양학과식품영양학과실내건축학과실내건축학과아동·가족학과아동·가족학과통합디자인학과통합디자인학과교육과학대학교육학과교육과학대학교육학과체육학과체육학과스포츠응용산업학과스포츠응용산업학과
의과대학
의학과
의과대학
의학과의과학과의과학과보건학과보건학과융합의학과융합의학과의료기기산업학과의료기기산업학과의생명시스템정보학과의생명시스템정보학과임상신약개발학과임상신약개발학과생체공학협동과정(폐과존치) 생체공학협동과정(폐과존치)언어병리학협동과정의료법·윤리학협동과정의료법·윤리학협동과정의학전산통계학협동과정의학전산통계학협

news.pdf 동일하게 진행

변수명:

loader_news, data_news, news_splits

In [26]:
# 코드 작성

news_path = os.path.join(os.getcwd(),'samples/rag_samples/news.pdf')
loader_news = load_pdf(news_path)

from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 데이터를 1000자 단위로 나눕니다. overlap은 100자로 설정합니다.
news_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

news_splits = news_splitter.split_documents(loader_news)

In [27]:
for i, split in enumerate(news_splits):
    print(f"Split {i+1}:------------------------------------")
    print(split)

Split 1:------------------------------------
page_content='1 
 
  
 
 
 
[2026 년 4 월 23 일] 
 
SK 하이닉스2026 년 1 분기 경영실적 발표 
➢ 
매출 52조 5,763억 원, 영업이익 37조 6,103억 원, 순이익 40조 3,459억 원 
➢ 
AI 수요 강세 속 고부가 제품 판매 확대… 사상 최대 분기 실적 달성 
➢ 
에이전틱 AI 시대, D램∙낸드 전방위 수요 증가… 신제품으로 선제 대응 계획  
➢ 
“수요 가시성 고려한 투자로 공급 안정성∙재무 건전성 동시 확보할 것” 
 
SK하이닉스가 올해 1분기 매출액 52조 5,763억 원, 영업이익 37조 6,103억 원(영
업이익률 72%), 순이익 40조 3,459억 원(순이익률 77%)의 경영실적을 기록했다
고 23일 밝혔다. (K-IFRS 기준) 
  
분기 기준으로 볼 때, 매출은 사상 최초로 50조 원을 돌파했으며, 영업이익과 
영업이익률 역시 각각 37.6조 원, 72%로 창사 이래 최고 기록을 달성했다. 영업이
익은 전분기 대비 2배 수준으로 급증하며 수익성 개선세를 뚜렷이 보여줬다. 
  
* 2025년 4분기 매출 32조 8,267억 원, 영업이익 19조 1,696억 원 
SK하이닉스는 "1분기는 계절적 비수기임에도 AI 인프라 투자 확대로 수요 강세
가 이어진 가운데, HBM·고용량 서버용 D램 모듈·eSSD 등 고부가가치 제품 판매를 
확대하며 실적 상승세를 이어갔다"고 설명했다. 
  
이 같은 실적 호조에 힘입어 1분기 말 현금성 자산은 전분기 말 대비 19.4조 
원 늘어난 54.3조 원을 기록했다. 반면, 차입금은 2.9조 원 감소한 19.3조 원을 기
록하며 35조 원의 순현금을 달성했다.' metadata={'source': 'd:\\autornd\\SK Autonomous R&D\\실습\\5일차\\samples/rag_samples/news.pdf', 'page': 0}
Split 2:------

In [28]:
all_splits.extend(news_splits)

In [29]:
all_splits

[Document(metadata={'producer': 'Hancom PDF 1.3.0.550', 'creator': 'Hwp 2024 13.0.0.3457', 'creationdate': '2026-06-17T16:37:51+09:00', 'author': 'DR', 'moddate': '2026-06-17T16:37:51+09:00', 'pdfversion': '1.4', 'source': 'd:\\autornd\\SK Autonomous R&D\\실습\\5일차\\samples/rag_samples/학칙.pdf', 'total_pages': 35, 'page': 0, 'page_label': '1'}, page_content='Ⅰ. 대학원 학칙  /  7\nⅠ. 대학원 학칙제정: 1974. 05. 18.개정(제122차): 2026. 06. 06.제1장 총칙제1조(목적) 대학원은 기독교 정신을 바탕으로 하여 창의적 이론과 과학적 방법을 탐구하고, 지도적 인격을 도야하여 인류 문화 향상에 기여함을 목적으로 한다.제2장 과정 및 정원제2조(과정) 대학원에는 석사학위를 취득하기 위한 석사학위과정, 박사학위를 취득하기 위한 박사학위과정, 석사학위과정을 거치지 않고 박사학위를 취득하기 위하여 석사학위과정과 박사학위과정이 통합된 과정(석·박사 통합과정, 이하 “통합과정”)을 두며, 학위과정 외에 학위를 수여하지 아니하는 연구과정을 둘 수 있다.제2조의2(협동과정) ① 대학원에 두는 학위과정으로 학과 외에 두 개 이상의 학과가 공동으로 설치·운영하는 협동과정(이하 “학과 간 협동과정”이라 한다)과 연구기관 또는 산업체와의 계약에 의하여 설치·운영하는 학·연·산 협동과정(이하 “학·연·산 협동과정”이라 한다)을 둘 수 있다.② 대학원장은 학과 간 협동과정의 운영실적을 2년마다 평가하여 대학원운영위원회 의결을 거쳐 존속여부를 결정하며, 이에 관한 사항은 따로 정한다.③ <삭제> 제2조의3 <삭제>제2조의4 <삭제>제2조의5(연계과정) 대학원에 두는 학위과정으로 본교 학부와

---
## 4. Embedding 

In [30]:
!uv add langchain_chroma langchain_openai

Resolved 258 packages in 3.03s
 Downloaded onnxruntime
 Downloaded chromadb
 Downloaded kubernetes
Prepared 21 packages in 10.56s
Installed 21 packages in 9.36s
 + bcrypt==5.0.0
 + build==1.5.0
 + chromadb==1.5.9
 + durationpy==0.10
 + flatbuffers==25.12.19
 + importlib-resources==7.1.0
 + kubernetes==36.0.2
 + langchain-chroma==1.1.0
 + mmh3==5.2.1
 + onnxruntime==1.27.0
 + opentelemetry-api==1.43.0
 + opentelemetry-exporter-otlp-proto-common==1.43.0
 + opentelemetry-exporter-otlp-proto-grpc==1.43.0
 + opentelemetry-proto==1.43.0
 + opentelemetry-sdk==1.43.0
 + opentelemetry-semantic-conventions==0.64b0
 + overrides==7.7.0
 + pybase64==1.4.3
 + pypika==0.51.1
 + pyproject-hooks==1.2.0
 + watchfiles==1.2.0


In [31]:
from langchain_openai import OpenAIEmbeddings 
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

In [32]:
embedding = OpenAIEmbeddings(model='text-embedding-3-large', api_key=OPENAI_API_KEY)

In [44]:
v = embedding.embed_query('연세대학교에서 논문제출 시한을 연장한 학생은 휴학할 수 있는가? 예외는 무엇인가?')

In [34]:
len(v)

3072

#### Embedding 모델

text-embedding-3-large : 높은 정확도 (*한글)

text-embedding-3-small : 비용 대비 성능이 뛰어남

---
## 4. 벡터 저장

다양한 저장소 존재 (FAISS, **Chroma**, Qdrant, ....)

In [35]:
from langchain_chroma import Chroma
import os

persist_directory = 'chroma_store'	

# 저장된 크로마 DB가 없다면 새로 만들기
if not os.path.exists(persist_directory):
    print("Creating new Chroma store")
    vectorstore = Chroma.from_documents(
        documents=all_splits,
        embedding=embedding,
        persist_directory=persist_directory
    )

else:
    print("Loading existing Chroma store")
    vectorstore = Chroma(		
        persist_directory=persist_directory, 
        embedding_function=embedding
    )

Creating new Chroma store


In [36]:
#FAISS 예시
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(all_splits, embeddings)

In [37]:
retriever = vectorstore.as_retriever(search_kwargs={"k":3})

docs = retriever.invoke("연세대학교에서 논문제출 시한을 연장한 학생은 휴학할 수 있는가? 예외는 무엇인가?")

In [38]:
docs

[Document(id='8d872d4b-5a9b-4636-a133-bdfd4d3ef2b4', metadata={'producer': 'Hancom PDF 1.3.0.550', 'creator': 'Hwp 2024 13.0.0.3457', 'creationdate': '2026-06-17T16:37:51+09:00', 'author': 'DR', 'moddate': '2026-06-17T16:37:51+09:00', 'pdfversion': '1.4', 'source': 'd:\\autornd\\SK Autonomous R&D\\실습\\5일차\\samples/rag_samples/학칙.pdf', 'total_pages': 35, 'page': 8, 'page_label': '9'}, page_content='제33조 <삭제> 제34조(심사판정) 논문심사 합격판정은 석사학위논문 심사위원 3분의 2 이상, 박사학위논문 심사위원 5분의 4 이상의 찬성으로 이루어진다.제35조(재심사) ① 학위논문 심사에 불합격 판정을 받은 학생은 적절한 수정 보완을 하여 재심사를 받을 수 있으며 이에 관한 사항은 따로 정한다.② <삭제>제36조(논문제출 시한) ① 석사학위과정 학생은 입학일로부터 4년 이내에, 박사학위과정 학생은 입학일로부터 7년 이내에, 통합과정 학생은 입학일로부터 8년 이내에 학위논문 심사에 합격하여 완성된 학위논문을 대학원에서 정한 절차에 따라 제출하여야 한다.② 위 제1항의 시한에는 등록학기만 산입하며, 휴학 및 제적기간은 산입하지 아니한다.③ 수료요건을 충족하고 합당한 사유가 있는 학생은 논문제출 시한 연장 사유서와 연구계획서를 제출하고 지도교수와 대학원장의 승인을 얻어 논문제출 시한을 2년 연장할 수 있다.④ 논문제출 시한 연장을 승인받은 기간에는 휴학할 수 없으나 다음 각호의 휴학은 예외로 처리할 수 있다.1. 입대휴학 2. 육아휴학3. 그 외에 대학원장이 인정하는 중대한 사유에 의한 일반휴학⑤ 「장애인 등에 대한 특수교육법」 제15조 제1항의 특수교육대

In [39]:
for d in docs:
    print(d)
    print('------')

page_content='제33조 <삭제> 제34조(심사판정) 논문심사 합격판정은 석사학위논문 심사위원 3분의 2 이상, 박사학위논문 심사위원 5분의 4 이상의 찬성으로 이루어진다.제35조(재심사) ① 학위논문 심사에 불합격 판정을 받은 학생은 적절한 수정 보완을 하여 재심사를 받을 수 있으며 이에 관한 사항은 따로 정한다.② <삭제>제36조(논문제출 시한) ① 석사학위과정 학생은 입학일로부터 4년 이내에, 박사학위과정 학생은 입학일로부터 7년 이내에, 통합과정 학생은 입학일로부터 8년 이내에 학위논문 심사에 합격하여 완성된 학위논문을 대학원에서 정한 절차에 따라 제출하여야 한다.② 위 제1항의 시한에는 등록학기만 산입하며, 휴학 및 제적기간은 산입하지 아니한다.③ 수료요건을 충족하고 합당한 사유가 있는 학생은 논문제출 시한 연장 사유서와 연구계획서를 제출하고 지도교수와 대학원장의 승인을 얻어 논문제출 시한을 2년 연장할 수 있다.④ 논문제출 시한 연장을 승인받은 기간에는 휴학할 수 없으나 다음 각호의 휴학은 예외로 처리할 수 있다.1. 입대휴학 2. 육아휴학3. 그 외에 대학원장이 인정하는 중대한 사유에 의한 일반휴학⑤ 「장애인 등에 대한 특수교육법」 제15조 제1항의 특수교육대상자는 「대학원 학칙」 제2조의8 제2항에 따라 합당한 사유가 있는 경우 대학원장의 승인을 받아 제3항의 연장 시한을 예외로 한다.제36조의2(논문제출) 학위논문 인준을 받은 학생은 대학원에서 정한 절차에 따라 완성된 학위논문을 제출하여야 한다.제8장 연구과정생, 외국인 학생, 공개강좌제37조(연구과정생) 연구과정생은 대학원 강의를 청강할 수 있으나 이수학점으로 인정되지 않으며 정해진 수업료를 납부하여야 하며, 이수자에게는 재학사실증명서를 발행할 수 있다.제38조(외국인 학생) 외국인으로서 본 대학원에 입학하고자 하는 자는 한국어 또는 영어 사용 능력이 일정한 수준에 도달하여야 하며, 별도의 입학전형에 합격할 경우 정원외로 입학을 허가할 수 있다. 단, 대학원에서 별도로 지정한 학과는

In [45]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

question = '석사학위과정 수업연한은?'
chat = ChatOpenAI(model="gpt-5.4-nano")
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})

In [ ]:
rag_prompt = ChatPromptTemplate.from_template("""
사용자의 질문에 대해 아래 context에 기반하여 답변하라

문맥:
{context}

질문: {question}
""")

### rag 답변 생성해보기

In [47]:
# 힌트
def format_docs(docs):
    return '\n\n'.join(d.page_content for d in docs)

In [73]:
result_docs = format_docs(docs)
print(result_docs)

제33조 <삭제> 제34조(심사판정) 논문심사 합격판정은 석사학위논문 심사위원 3분의 2 이상, 박사학위논문 심사위원 5분의 4 이상의 찬성으로 이루어진다.제35조(재심사) ① 학위논문 심사에 불합격 판정을 받은 학생은 적절한 수정 보완을 하여 재심사를 받을 수 있으며 이에 관한 사항은 따로 정한다.② <삭제>제36조(논문제출 시한) ① 석사학위과정 학생은 입학일로부터 4년 이내에, 박사학위과정 학생은 입학일로부터 7년 이내에, 통합과정 학생은 입학일로부터 8년 이내에 학위논문 심사에 합격하여 완성된 학위논문을 대학원에서 정한 절차에 따라 제출하여야 한다.② 위 제1항의 시한에는 등록학기만 산입하며, 휴학 및 제적기간은 산입하지 아니한다.③ 수료요건을 충족하고 합당한 사유가 있는 학생은 논문제출 시한 연장 사유서와 연구계획서를 제출하고 지도교수와 대학원장의 승인을 얻어 논문제출 시한을 2년 연장할 수 있다.④ 논문제출 시한 연장을 승인받은 기간에는 휴학할 수 없으나 다음 각호의 휴학은 예외로 처리할 수 있다.1. 입대휴학 2. 육아휴학3. 그 외에 대학원장이 인정하는 중대한 사유에 의한 일반휴학⑤ 「장애인 등에 대한 특수교육법」 제15조 제1항의 특수교육대상자는 「대학원 학칙」 제2조의8 제2항에 따라 합당한 사유가 있는 경우 대학원장의 승인을 받아 제3항의 연장 시한을 예외로 한다.제36조의2(논문제출) 학위논문 인준을 받은 학생은 대학원에서 정한 절차에 따라 완성된 학위논문을 제출하여야 한다.제8장 연구과정생, 외국인 학생, 공개강좌제37조(연구과정생) 연구과정생은 대학원 강의를 청강할 수 있으나 이수학점으로 인정되지 않으며 정해진 수업료를 납부하여야 하며, 이수자에게는 재학사실증명서를 발행할 수 있다.제38조(외국인 학생) 외국인으로서 본 대학원에 입학하고자 하는 자는 한국어 또는 영어 사용 능력이 일정한 수준에 도달하여야 하며, 별도의 입학전형에 합격할 경우 정원외로 입학을 허가할 수 있다. 단, 대학원에서 별도로 지정한 학과는 외국인을 위한

16
학

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# retriever → context 문자열 → rag_prompt → LLM → 문자열
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | chat
    | StrOutputParser()
)

answer = rag_chain.invoke(question)
print(answer)

In [ ]:
# 다른 질문으로도 바로 테스트
rag_chain.invoke("석사학위과정 수업연한은?")

## Retriever -> Tool

In [64]:
from langchain_core.tools import create_retriever_tool
from langchain_core.messages import HumanMessage

llm = ChatOpenAI(model="gpt-5.4-nano")
search_tool = create_retriever_tool(
    retriever,
    name = 'search_pdf_documents',
    description = ' pdf samples 안의 문서함에서 질문과 관련된 내용을 검색합니다.'
)

In [65]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001A673E6BE00>, search_kwargs={'k': 3})

In [66]:
retriever.invoke('석사 수업연한')

[Document(id='89926d15-8e45-41ca-9bfd-2ff4c08fa42d', metadata={'producer': 'Hancom PDF 1.3.0.550', 'creator': 'Hwp 2024 13.0.0.3457', 'creationdate': '2026-06-17T16:37:51+09:00', 'author': 'DR', 'moddate': '2026-06-17T16:37:51+09:00', 'pdfversion': '1.4', 'source': 'd:\\autornd\\SK Autonomous R&D\\실습\\5일차\\samples/rag_samples/학칙.pdf', 'total_pages': 35, 'page': 1, 'page_label': '2'}, page_content='③ <삭제> 제2조의7(수업연한) ① 대학원 각 학위과정의 수업연한은 다음 각호와 같다.1. 석사학위과정: 2년(4학기)2. 박사학위과정: 2년(4학기)3. 통합과정: 3년(6학기)② 제1항의 규정에도 불구하고 학위수여 자격 요건을 충족한 학생은 수업연한을 1학기 단축할 수 있다. 단, 통합과정 및 석사학위논문 대체실적을 통해 학위를 취득하고자 하는 학생은 제외한다.제2조의8(재학연한) ① 대학원 각 학위과정의 재학연한은 다음 각호와 같으며, 이를 초과할 수 없다. 단, 휴학 및 제적 기간은 재학연한에 산입하지 아니한다. 1. 석사학위과정: 4년(8학기)2. 박사학위과정: 7년(14학기)3. 통합과정: 8년(16학기)② 「장애인 등에 대한 특수교육법」 제15조 제1항의 특수교육대상자로서 절차에 따라 학적에 구분표시 되어 있는 학생에 대하여는 재학연한을 제한하지 아니한다.제2조의9(융합전공) ① 대학원 내 두 개 이상의 학과가 융합적 교과과정(융합전공)을 공동 운영할 수 있다.② 융합전공의 운영에 관한 사항은 별도로 정한다.제2조의10(학생설계전공) ① 새로운 학문수요의 충족과 자기 주도적 학습능력의 향상을 위하여 학생설계전공을 둘 수 있다.② 학생설계전공의

In [67]:
search_tool.invoke('석사 수업연한')

'③ <삭제> 제2조의7(수업연한) ① 대학원 각 학위과정의 수업연한은 다음 각호와 같다.1. 석사학위과정: 2년(4학기)2. 박사학위과정: 2년(4학기)3. 통합과정: 3년(6학기)② 제1항의 규정에도 불구하고 학위수여 자격 요건을 충족한 학생은 수업연한을 1학기 단축할 수 있다. 단, 통합과정 및 석사학위논문 대체실적을 통해 학위를 취득하고자 하는 학생은 제외한다.제2조의8(재학연한) ① 대학원 각 학위과정의 재학연한은 다음 각호와 같으며, 이를 초과할 수 없다. 단, 휴학 및 제적 기간은 재학연한에 산입하지 아니한다. 1. 석사학위과정: 4년(8학기)2. 박사학위과정: 7년(14학기)3. 통합과정: 8년(16학기)② 「장애인 등에 대한 특수교육법」 제15조 제1항의 특수교육대상자로서 절차에 따라 학적에 구분표시 되어 있는 학생에 대하여는 재학연한을 제한하지 아니한다.제2조의9(융합전공) ① 대학원 내 두 개 이상의 학과가 융합적 교과과정(융합전공)을 공동 운영할 수 있다.② 융합전공의 운영에 관한 사항은 별도로 정한다.제2조의10(학생설계전공) ① 새로운 학문수요의 충족과 자기 주도적 학습능력의 향상을 위하여 학생설계전공을 둘 수 있다.② 학생설계전공의 설치 및 이수요건은 별도로 정한다.제3조(입학정원) ① 대학원에 두는 학위과정별 입학정원([별표 1])은 대학원운영위원회에서 대학별로 정한다. 단, 통합과정 입학정원은 박사학위과정의 입학정원에 산정된다.② 다음 각호의 어느 하나에 해당하는 학생의 정원은 제1항의 규정에 의한 정원 외에 따로 있는 것으로 본다.1. 대학원에 입학·편입학 또는 재입학하는 다음 각 목의 학생가. 교육부령으로 정하는 위탁학생나. 북한이탈주민다. 부모가 모두 외국인인 외국인라. 외국에서 우리나라 초·중등교육과 대학교육에 상응하는 교육과정을 전부 이수한 재외국민 및 외국인2. 「고등교육법 시행령」 제13조 제1항 제1호 또는 제2호에 따라 외국대학과 공동으로 운영하는 대학원 교육과정에 입학하는 학생3. 「고등교육법 시행령」 제28조\

In [68]:
tools = [search_tool]
tool_dict = {'search_pdf_documents':search_tool}

llm_with_tools = llm.bind_tools(tools)

messages = [
    HumanMessage('석사학위과정 수업연한은?')
]

response = llm_with_tools.invoke('석사학위과정 수업연한은?')
messages.append(response)

In [69]:
response.tool_calls

[{'name': 'search_pdf_documents',
  'args': {'query': '석사학위과정 수업연한'},
  'id': 'call_Japeguqqn5mQiJRZNCxE6nRQ',
  'type': 'tool_call'}]

In [70]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # (7) tool_dict를 사용하여 도구 함수를 선택
    print(tool_call["args"]) # (8) 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # (9) 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

{'query': '석사학위과정 수업연한'}


[HumanMessage(content='석사학위과정 수업연한은?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 156, 'total_tokens': 183, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DuvvPc82f7ph3ra65gqx61OdJYMdY', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f02fa-8f09-7bb3-ba49-b631592013e0-0', tool_calls=[{'name': 'search_pdf_documents', 'args': {'query': '석사학위과정 수업연한'}, 'id': 'call_Japeguqqn5mQiJRZNCxE6nRQ', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 156, 'output_tokens': 27, 'total_tokens': 183, 'input_token_details': {'audi

In [71]:
result_final = llm_with_tools.invoke(messages)
print(result_final.content)

석사학위과정의 **수업연한은 2년(4학기)** 입니다.
